# Distance ou angle ?

| Module | Durée | Prérequis |
|---|---|---|
| M1 – Vecteurs et ressemblance | 75 min | M0 |

In [ ]:
# Style graphique Liouaï : couleurs du logo, axes sobres
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap

LIOUAI = {"navy": "#243743", "vert": "#28B78D", "vert_texte": "#167A5C", "gris": "#A1A9AE",
          "gris_texte": "#5C6870", "jaune": "#E5C800", "fond": "#F3F5F6"}
plt.rcParams.update({
    "axes.prop_cycle": plt.cycler(color=[LIOUAI["vert"], LIOUAI["navy"], LIOUAI["gris"],
                                         LIOUAI["jaune"], LIOUAI["vert_texte"], LIOUAI["gris_texte"]]),
    "axes.edgecolor": LIOUAI["gris"], "axes.labelcolor": LIOUAI["navy"],
    "axes.titlecolor": LIOUAI["navy"], "axes.titleweight": "bold",
    "axes.spines.top": False, "axes.spines.right": False,
    "xtick.color": LIOUAI["gris_texte"], "ytick.color": LIOUAI["gris_texte"],
    "text.color": LIOUAI["navy"], "grid.color": "#E1E6E9", "legend.frameon": False,
    "font.family": "sans-serif", "font.sans-serif": ["Roboto", "Helvetica", "Arial", "DejaVu Sans"],
    "figure.dpi": 100,
})
# Dégradé de marque : gris clair -> vert -> navy (et sa version inversée)
if "liouai" not in mpl.colormaps:
    _cmap = LinearSegmentedColormap.from_list("liouai", [LIOUAI["fond"], LIOUAI["vert"], LIOUAI["navy"]])
    mpl.colormaps.register(_cmap)
    mpl.colormaps.register(_cmap.reversed())

## ① Définir la question

Une petite plateforme d'écoute musicale veut suggérer à chaque membre « les auditeurs qui vous ressemblent ». Pour chaque membre, elle connaît le nombre d'heures d'écoute par semaine dans quatre genres : **pop, rock, jazz, classique**.

| Membre | pop | rock | jazz | classique |
|---|---|---|---|---|
| Alice | 4 | 2 | 0 | 0 |
| Bruno | 40 | 20 | 0 | 0 |
| Chloé | 3 | 0 | 3 | 0 |
| David | 0 | 1 | 5 | 6 |
| Emma | 1 | 1 | 1 | 1 |
| Farid | 12 | 6 | 1 | 0 |

**Activité** : « Se ressembler » peut vouloir dire deux choses différentes. Lesquelles ? Formulez chacune comme une question précise.

Indiquez vos réponses et justifiez-les.

## ② Estimer avant de calculer

In [ ]:
pari_plus_proche_alice = "Chloé"     # selon vous, qui ressemble le plus à Alice ? (modifiez)

## ③ Abstraire : trois outils pour un même concept

Chaque membre est un **vecteur de dimension 4**. Le concept est « mesurer la ressemblance entre deux vecteurs ». Trois outils sont couramment utilisés :

| Outil | Formule | Ce qu'il mesure |
|---|---|---|
| **Distance euclidienne** | $\lVert \mathbf{u} - \mathbf{v} \rVert = \sqrt{\sum_i (u_i - v_i)^2}$ | la longueur de l'écart, « à vol d'oiseau » |
| **Distance de Manhattan** | $\sum_i \lvert u_i - v_i \rvert$ | la somme des écarts, genre par genre |
| **Similarité cosinus** | $\cos\alpha = \dfrac{\mathbf{u} \cdot \mathbf{v}}{\lVert \mathbf{u} \rVert \, \lVert \mathbf{v} \rVert}$ | le cosinus de l'**angle** $\alpha$ entre les vecteurs, entre 0 et 1 ici |

Le **produit scalaire** $\mathbf{u} \cdot \mathbf{v} = \sum_i u_i v_i$ est grand quand les deux vecteurs ont de grandes valeurs aux mêmes endroits. Divisé par les longueurs, il ne dépend plus que de l'**orientation** : c'est le cosinus de l'angle.

Une distance vaut 0 pour deux vecteurs identiques ; une similarité vaut 1 pour deux vecteurs de même direction. Attention à ne pas les confondre.

![Schéma : Alice et Bruno ont la même direction mais sont éloignés ; Alice et Chloé sont proches mais forment un angle](../../assets/schemas/schema-distance-angle.svg)

**Activité** : Calculez la distance euclidienne et la similarité cosinus entre Alice $(4, 2, 0, 0)$ et Bruno $(40, 20, 0, 0)$.

Indiquez vos réponses et justifiez-les.

## ④ Calculer

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

genres = ["pop", "rock", "jazz", "classique"]
noms = ["Alice", "Bruno", "Chloé", "David", "Emma", "Farid"]
X = np.array([
    [ 4,  2, 0, 0],
    [40, 20, 0, 0],
    [ 3,  0, 3, 0],
    [ 0,  1, 5, 6],
    [ 1,  1, 1, 1],
    [12,  6, 1, 0],
], dtype=float)

def euclidienne(u, v):
    return np.linalg.norm(u - v)

def manhattan(u, v):
    return np.abs(u - v).sum()

def cosinus(u, v):
    return (u @ v) / (np.linalg.norm(u) * np.linalg.norm(v))

alice, bruno = X[0], X[1]
print(f"Alice–Bruno : euclidienne {euclidienne(alice, bruno):.2f}, Manhattan {manhattan(alice, bruno):.0f}, cosinus {cosinus(alice, bruno):.3f}")

In [ ]:
def matrice(mesure, X):
    n = len(X)
    return np.array([[mesure(X[i], X[j]) for j in range(n)] for i in range(n)])

mesures = {"euclidienne": euclidienne, "Manhattan": manhattan, "cosinus": cosinus}
fig, axes = plt.subplots(1, 3, figsize=(14, 4.2))
for ax, (nom, f) in zip(axes, mesures.items()):
    Mx = matrice(f, X)
    im = ax.imshow(Mx, cmap="liouai_r" if nom != "cosinus" else "liouai", vmin=Mx.min(), vmax=Mx.max())
    ax.set_xticks(range(6), noms, rotation=45); ax.set_yticks(range(6), noms)
    for i in range(6):
        for j in range(6):
            r, g, b, _ = im.cmap(im.norm(Mx[i, j]))
            fonce = 0.299 * r + 0.587 * g + 0.114 * b < 0.55          # texte blanc sur case foncée
            ax.text(j, i, f"{Mx[i, j]:.2f}" if nom == "cosinus" else f"{Mx[i, j]:.0f}",
                    ha="center", va="center", fontsize=8, color="white" if fonce else LIOUAI["navy"])
    ax.set_title(nom)
plt.suptitle("Plus la case est foncée, plus les deux membres se ressemblent selon la mesure")
plt.tight_layout(); plt.show()

In [ ]:
def plus_proche(i, mesure, X, plus_grand_est_mieux=False):
    scores = [(mesure(X[i], X[j]), noms[j]) for j in range(len(X)) if j != i]
    return sorted(scores, reverse=plus_grand_est_mieux)[0][1]

for nom, f in mesures.items():
    print(f"{nom:12s}: le plus proche d'Alice est", plus_proche(0, f, X, plus_grand_est_mieux=(nom == "cosinus")))

## ⑤ Interpréter

**Activité** :

1. Selon la distance euclidienne, qui ressemble le plus à Alice ? Et selon le cosinus ? Comparez avec votre pari.
2. Bruno écoute **dix fois plus** qu'Alice, dans les mêmes proportions. Laquelle des deux mesures « voit » qu'ils ont les mêmes goûts ? Laquelle « voit » qu'ils n'ont pas la même intensité d'écoute ?
3. Revenez aux deux questions de l'étape ①. Quelle mesure correspond à chacune ?

Indiquez vos réponses et justifiez-les.

### Normaliser : le lien entre distance et angle

Si l'on divise chaque vecteur par sa norme, ils ont tous une longueur 1 : seule leur direction compte. Pour deux vecteurs normalisés $\hat{\mathbf{u}}$ et $\hat{\mathbf{v}}$, la distance et le cosinus sont liés :

$$
\lVert \hat{\mathbf{u}} - \hat{\mathbf{v}} \rVert = \sqrt{2 - 2\cos\alpha}
$$

La distance entre vecteurs normalisés donne donc le même classement que le cosinus.

In [ ]:
Xn = X / np.linalg.norm(X, axis=1, keepdims=True)
for j in range(1, 6):
    d = euclidienne(Xn[0], Xn[j]); c = cosinus(X[0], X[j])
    print(f"Alice–{noms[j]:6s} distance normalisée {d:.3f}   cosinus {c:.3f}   vérification √(2 − 2·cos) = {np.sqrt(2 - 2 * c):.3f}")

## ⑥ Critiquer et généraliser

- **Comparer les outils.** Complétez le tableau : pour chaque besoin, quelle mesure choisir ?

| Besoin | Mesure conseillée | Pourquoi |
|---|---|---|
| suggérer des auditeurs aux goûts proches | … | … |
| repérer les gros consommateurs semblables | … | … |
| comparer des textes de longueurs très différentes | … | … |

- **Hypothèse fragile.** On a supposé que les heures d'écoute reflètent les goûts. Citez une situation où ce n'est pas le cas (écoute en fond sonore, compte partagé…). Comment cela fausse-t-il les ressemblances ?
- **Cas limite.** Que renvoie `cosinus` pour un membre qui n'a rien écouté, avec le vecteur (0, 0, 0, 0) ? Corrigez la fonction pour traiter ce cas.

In [ ]:
# Que se passe-t-il pour un vecteur nul ?
with np.errstate(invalid="ignore", divide="ignore"):
    print(cosinus(np.zeros(4), alice))

In [ ]:
def cosinus_sur(u, v):
    # À compléter : renvoyer 0.0 si l'un des vecteurs est nul
    ...

## Corrigé

*À n'ouvrir qu'après avoir cherché.*

In [ ]:
def cosinus_sur(u, v):
    nu, nv = np.linalg.norm(u), np.linalg.norm(v)
    if nu == 0 or nv == 0:
        return 0.0
    return float(u @ v / (nu * nv))

print(cosinus_sur(np.zeros(4), alice), cosinus_sur(alice, bruno))
print("Votre pari :", pari_plus_proche_alice)

## Bilan — auto-évaluation

Pour chaque compétence travaillée dans ce notebook, choisissez un niveau : **0** non observé · **1** guidé · **2** autonome · **3** transfert.
La cellule enregistre votre bilan dans `progression.json`, à la racine du site.

In [ ]:
import json, datetime, pathlib

bilan = {
    "Comparer les mérites des outils dans le contexte": 1,
    "Connaître des outils comparables": 1,
    "Quantifier la validité et l'impact des hypothèses": 1,
    "Repérer les traits intéressants du résultat": 1,
}   # modifiez les niveaux

ici = pathlib.Path.cwd()
racine = next((p for p in [ici, *ici.parents] if (p / "myst.yml").exists()), ici)
fichier = racine / "progression.json"
historique = json.loads(fichier.read_text(encoding="utf-8")) if fichier.exists() else []
historique.append({"notebook": "1.3-distance-ou-angle", "date": datetime.date.today().isoformat(), "niveaux": bilan})
fichier.write_text(json.dumps(historique, ensure_ascii=False, indent=2), encoding="utf-8")
print(f"Bilan enregistré ({len(historique)} entrée(s) dans {fichier.name}).")